# Benchmark: Qwen2.5-Coder-7B fine-tuned on OpenCodeInstruct HQ 10K

Đánh giá **pass@1** của `hoangnh39/qwen2.5-coder-7b-opencode-hq10k-3ep` (eval lần lượt các branch `epoch-1/2/3` trong `FT_REVISIONS` trong một lần chạy) trên **HumanEval** và **HumanEval+** (EvalPlus) với **few-shot prompt**, cùng prompt/hàm chấm điểm như `src/code-generation-kaggle-2.ipynb` để so sánh công bằng với Qwen base / instruct.

**Khác biệt chính so với notebook benchmark base:** model được fine-tune trên format **ChatML** (`<|im_start|>user ... <|im_end|>\n<|im_start|>assistant`), nên prompt few-shot được bọc trong ChatML và generation dừng ở `<|im_end|>`. Đặt `PROMPT_FORMAT = "raw"` để chạy kiểu completion thuần (giống cách eval Qwen base) nếu muốn so sánh.

**Trước khi chạy (Kaggle):**
1. Settings → Accelerator: GPU (T4 x2 / P100). Settings → Internet: On.
2. Add-ons → Secrets: thêm `HF_TOKEN` (repo model đang private, token cần quyền đọc) và attach vào notebook.

**Thay đổi so với lần chạy đầu (sau khi phân tích kết quả):**
- Thêm các dòng `import` ở đầu prompt vào code trước khi chạy test (model hay chép chữ ký hàm nhưng bỏ `from typing import List` → `NameError`).
- `MAX_NEW_TOKENS` 512 → 1024; timeout chạy test 5s → 15s; `NUM_SAMPLES` 1 → 5 (sinh batch bằng `num_return_sequences`) để pass@1 bớt nhiễu.
- `BASELINES` được chấm lại offline với đúng harness này, nên con số khác `data/summary.json` cũ một chút.

In [1]:
# Không đụng torch/torchvision (Kaggle đã ghép sẵn đúng bản). Nếu sau khi cài lại báo lỗi import
# transformers/peft thì Restart Session rồi chạy lại từ cell tiếp theo.
!pip install -q -U peft bitsandbytes accelerate datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 36.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.


## 1. Config

In [2]:
import gc
import math
import multiprocessing
import json
import re
import os
import time
from tqdm import tqdm
import torch
from datasets import load_dataset
from huggingface_hub import login, hf_hub_download
from huggingface_hub.utils import EntryNotFoundError
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_MODEL_ID  = "Qwen/Qwen2.5-Coder-7B"
FT_MODEL_ID    = "hoangnh39/qwen2.5-coder-7b-opencode-hq10k-3ep"
FT_REVISIONS   = ["epoch-1", "epoch-2", "epoch-3"]   # các branch adapter theo epoch sẽ eval lần lượt trong 1 lần chạy
def model_key(rev): return f"qwen_ft_opencode_hq10k_3ep_{rev.replace('-', '')}"   # vd qwen_ft_opencode_hq10k_3ep_epoch3
def adapter_name(rev): return rev.replace("-", "_")
MODEL_KEY      = model_key(FT_REVISIONS[0])   # được cập nhật trong vòng lặp ở mục 6

NUM_SAMPLES    = 5        # samples / problem (pass@1 = trung bình trên các mẫu, ít nhiễu hơn n=1)
K_VALUE        = 1        # pass@1
MAX_NEW_TOKENS = 1024     # 512 làm cụt các bài có docstring dài (vd HumanEval/81, 129)
TEMPERATURE    = 0.2      # giống các benchmark trước
TOP_P          = 0.95

PROMPT_FORMAT  = "chatml"           # "chatml" (khớp lúc train) | "raw" (completion thuần)
RUN_TEMPLATES  = ["few_shot"]       # thêm "zero_shot" nếu muốn chạy cả hai
LIMIT          = None               # vd 10 để smoke test; None = full 164 problems / dataset
SKIP_IF_EXISTS = True               # bỏ qua run đã có file kết quả (resume khi session bị ngắt)

EXEC_TIMEOUT   = 15        # giây/bài khi chạy test (5s làm fail oan vài bài HumanEval+)
RESULT_SUFFIX  = f"_n{NUM_SAMPLES}"   # để không bị SKIP_IF_EXISTS nhầm với file n=1 cũ

RESULTS_DIR = "/kaggle/working/data"
os.makedirs(RESULTS_DIR, exist_ok=True)

DATASETS = {
    "humaneval":      ("openai/openai_humaneval", "test"),
    "humaneval_plus": ("evalplus/humanevalplus", "test"),
}

# Baseline (n=1) chấm lại offline từ data/results_qwen_*_few_shot.json với đúng harness mới
# (timeout 15s, có thêm import của prompt) -> so sánh công bằng với model fine-tune.
BASELINES = {
    "qwen_base/humaneval/few_shot":          0.7683,
    "qwen_base/humaneval_plus/few_shot":     0.6951,
    "qwen_instruct/humaneval/few_shot":      0.8780,
    "qwen_instruct/humaneval_plus/few_shot": 0.8232,
}

hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
hf_token = hf_token or os.environ.get("HF_TOKEN")
if not hf_token:
    from getpass import getpass
    hf_token = getpass("Enter your Hugging Face token (HF_TOKEN): ")
login(hf_token)

## 2. Prompt Templates (giữ nguyên như benchmark trước)

In [3]:
ZERO_SHOT_PROMPT = """You are an expert Python programmer. Write the implementation for the following problem.
Do not output anything else but the code block.

Requirements:
- Output only a single Python code block.
- Include all necessary imports explicitly (e.g., from typing import List, Optional, etc.).
- Do not assume any helper functions exist — define everything you use.
- The solution must be self-contained and executable.

Problem:
{prompt}

Code:
"""

FEW_SHOT_PROMPT = """You are an expert Python programmer. You will be provided some examples, and then a problem to solve.
Do not output anything else but the code block.

Requirements:
- Output only a single Python code block.
- Include all necessary imports explicitly (e.g., from typing import List, Optional, etc.).
- Do not assume any helper functions exist — define everything you use.
- The solution must be self-contained and executable.

Example 1:
Problem:
def add(a, b):
    \"\"\"Return the sum of a and b.\"\"\"

Code:
```python
def add(a, b):
    \"\"\"Return the sum of a and b.\"\"\"
    return a + b
```

Example 2:
Problem:
def is_palindrome(s: str) -> bool:
    \"\"\"Check if a string is a palindrome.\"\"\"

Code:
```python
def is_palindrome(s: str) -> bool:
    \"\"\"Check if a string is a palindrome.\"\"\"
    return s == s[::-1]
```

Problem:
{prompt}

Code:
"""

ALL_TEMPLATES = {"zero_shot": ZERO_SHOT_PROMPT, "few_shot": FEW_SHOT_PROMPT}
PROMPT_TEMPLATES = {k: ALL_TEMPLATES[k] for k in RUN_TEMPLATES}

## 3. Helper Functions (execution, pass@k, cleanup)

In [4]:
def run_eval(code, test_code, queue):
    local_scope = {}
    try:
        exec(code, local_scope)
        exec(test_code, local_scope)
        queue.put(True)
    except BaseException:
        queue.put(False)

def exec_code(code, test_code, timeout=5):
    """Executes the code and the test, returning True if passes, False otherwise (sandboxed in a subprocess)."""
    queue = multiprocessing.Queue()
    p = multiprocessing.Process(target=run_eval, args=(code, test_code, queue))
    p.start()
    p.join(timeout)

    if p.is_alive():
        p.terminate()
        p.join()
        return False

    return queue.get() if not queue.empty() else False

def calculate_pass_at_k(n, c, k):
    if n - c < k:
        return 1.0
    if c == 0:
        return 0.0
    try:
        return 1.0 - (math.comb(n - c, k) / math.comb(n, k))
    except ValueError:
        return 0.0

def clean_code(generated_text):
    match = re.search(r"```python\s*(.*?)\s*```", generated_text, re.DOTALL)
    if match:
        return match.group(1)
    match = re.search(r"```\s*(.*?)\s*```", generated_text, re.DOTALL)
    if match:
        return match.group(1)
    # Model đôi khi mở ```python mà không đóng (bị cắt ở max_new_tokens)
    match = re.search(r"```(?:python)?\s*(.*)", generated_text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return generated_text.strip()

def prompt_imports(prompt):
    """Các dòng import ở đầu prompt (vd `from typing import List`). Model hay chép chữ ký hàm
    nhưng bỏ import -> NameError dù logic đúng; cách eval chuẩn của HumanEval giữ lại prompt này."""
    return "\n".join(l for l in prompt.splitlines() if re.match(r"^(import |from \S+ import )", l))

def save_results(results, filename):
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=4)

## 4. Load base + các LoRA adapter theo epoch

Base load **một lần** (4-bit NF4, compute fp16 vì T4 không có bf16, giống các benchmark trước), sau đó nạp từng adapter `epoch-N` thành một adapter riêng (`peft` named adapters). Mục 6 chỉ cần `set_adapter(...)` để đổi epoch, không phải load lại 7B.


In [5]:
from peft import PeftModel

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, quantization_config=bnb_config, device_map="auto")

first, *rest = FT_REVISIONS
model = PeftModel.from_pretrained(base, FT_MODEL_ID, revision=first, adapter_name=adapter_name(first), token=hf_token)
for rev in rest:
    model.load_adapter(FT_MODEL_ID, adapter_name=adapter_name(rev), revision=rev, token=hf_token)
model.set_adapter(adapter_name(first))
model.eval()
print("Adapters:", list(model.peft_config), "| active:", model.active_adapter)

IM_END_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")
STOP_IDS = [IM_END_ID, tokenizer.eos_token_id]
print("<|im_end|> id:", IM_END_ID, "| eos:", tokenizer.eos_token_id, "| GPU mem:", f"{torch.cuda.memory_allocated()/1e9:.1f} GB")

config.json:   0%|          | 0.00/668 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.31k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B /  162MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B /  162MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

Adapters: ['epoch_1', 'epoch_2', 'epoch_3'] | active: epoch_1
<|im_end|> id: 151645 | eos: 151643 | GPU mem: 1.5 GB


## 5. Generator

In [6]:
from transformers import StoppingCriteria, StoppingCriteriaList

class StopOnCodeBlockEnd(StoppingCriteria):
    """Dừng ngay khi khối ``` đầu tiên được đóng. clean_code chỉ lấy khối đầu nên không đổi kết quả,
    nhưng bỏ qua phần giải thích dài mà model học từ OpenCodeInstruct (tiết kiệm tới ~10x thời gian)."""
    def __init__(self, tokenizer, prompt_len):
        self.tokenizer, self.prompt_len = tokenizer, prompt_len

    def __call__(self, input_ids, scores, **kwargs):
        # Trả về cờ dừng cho TỪNG chuỗi trong batch (num_return_sequences > 1)
        texts = self.tokenizer.batch_decode(input_ids[:, self.prompt_len:], skip_special_tokens=True)
        done = []
        for t in texts:
            first = t.find("```")
            done.append(first != -1 and t.find("```", first + 3) != -1)
        return torch.tensor(done, device=input_ids.device)


class QwenHFGenerator:
    def __init__(self, model, tokenizer, prompt_format="chatml"):
        self.model = model
        self.tokenizer = tokenizer
        self.prompt_format = prompt_format

    def _build_inputs(self, full_prompt):
        if self.prompt_format == "chatml":
            # Đúng format lúc train (không system prompt)
            full_prompt = f"<|im_start|>user\n{full_prompt}<|im_end|>\n<|im_start|>assistant\n"
        return self.tokenizer(full_prompt, return_tensors="pt").to(self.model.device)

    def generate(self, prompt_template, task_prompt, num_samples=1):
        full_prompt = prompt_template.format(prompt=task_prompt)
        inputs = self._build_inputs(full_prompt)

        prompt_len = inputs["input_ids"].shape[1]
        stopping = StoppingCriteriaList([StopOnCodeBlockEnd(self.tokenizer, prompt_len)])

        def _gen(n):
            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=MAX_NEW_TOKENS,
                    temperature=TEMPERATURE,
                    top_p=TOP_P,
                    do_sample=True,
                    num_return_sequences=n,      # sinh n mẫu trong 1 lần gọi: nhanh hơn nhiều so với n lần riêng lẻ
                    eos_token_id=STOP_IDS,
                    stopping_criteria=stopping,
                    pad_token_id=self.tokenizer.eos_token_id,
                )
            return [self.tokenizer.decode(o[prompt_len:], skip_special_tokens=True) for o in outputs]

        try:
            return _gen(num_samples)
        except Exception as e:                   # vd OOM -> thử lại từng mẫu một
            print(f"Batched generation failed ({e}); falling back to one-by-one")
            torch.cuda.empty_cache()
            samples = []
            for _ in range(num_samples):
                try:
                    samples.extend(_gen(1))
                except Exception as e2:
                    print(f"Error during generation: {e2}")
                    samples.append("")
            return samples

generator = QwenHFGenerator(model, tokenizer, prompt_format=PROMPT_FORMAT)

# Smoke test 1 prompt (in ra thời gian để biết tốc độ thật)
_t = time.time()
_out = generator.generate(FEW_SHOT_PROMPT, "def fib(n: int) -> int:\n    \"\"\"Return the n-th Fibonacci number.\"\"\"\n")[0]
print(_out)
print(f"--- {time.time()-_t:.1f}s, {len(tokenizer(_out)['input_ids'])} tokens")

```python
def fib(n: int) -> int:
    """Return the n-th Fibonacci number."""
    if n <= 0:
        return 0
    elif n == 1:
        return 1
    else:
        a, b = 0, 1
        for _ in range(2, n + 1):
            a, b = b, a + b
        return b
```
--- 13.3s, 88 tokens


## 6. Evaluation Loop

In [7]:
def evaluate_dataset(dataset, dataset_key, template_name, prompt_template):
    print(f"\n===== Model [{MODEL_KEY}] | Dataset [{dataset_key}] | Template [{template_name}] | Format [{PROMPT_FORMAT}] =====")
    results = []
    total_pass_at_k = 0

    for item in tqdm(dataset):
        entry_point = item["entry_point"]

        start_time = time.time()
        raw_samples = generator.generate(prompt_template, item["prompt"], num_samples=NUM_SAMPLES)
        time_taken = time.time() - start_time

        correct_count = 0
        cleaned_codes = []
        for raw_text in raw_samples:
            code = clean_code(raw_text)
            full_code = prompt_imports(item["prompt"]) + "\n" + code
            eval_code = full_code + "\n" + item["test"] + f"\ncheck({entry_point})\n"
            passed = exec_code(full_code, eval_code, timeout=EXEC_TIMEOUT)
            correct_count += int(passed)
            cleaned_codes.append({"raw": raw_text, "extracted": code, "passed": passed})

        pass_at_k = calculate_pass_at_k(NUM_SAMPLES, correct_count, K_VALUE)
        total_pass_at_k += pass_at_k
        results.append({
            "task_id": item["task_id"],
            "task_name": entry_point,
            "samples": cleaned_codes,
            "correct_count": correct_count,
            "n": NUM_SAMPLES,
            "pass@k": pass_at_k,
            "time_taken": time_taken,
        })

    avg = total_pass_at_k / len(dataset)
    avg_time = sum(r["time_taken"] for r in results) / len(results)
    print(f"[{MODEL_KEY}][{dataset_key}][{template_name}] Avg pass@{K_VALUE}: {avg:.4f} - Avg time: {avg_time:.2f}s/it")

    filename = os.path.join(RESULTS_DIR, f"results_{MODEL_KEY}_{dataset_key}_{template_name}{RESULT_SUFFIX}.json")
    save_results(results, filename)
    print(f"Results saved to {filename}")
    return avg


summary = {}
for rev in FT_REVISIONS:
    model.set_adapter(adapter_name(rev))      # đổi sang adapter của epoch này (generator dùng chung `model`)
    MODEL_KEY = model_key(rev)                # evaluate_dataset đọc biến global này để đặt tên file/log
    print(f"\n########## {FT_MODEL_ID}@{rev} -> {MODEL_KEY} (active adapter: {model.active_adapter}) ##########")
    for dataset_key, (hf_path, split) in DATASETS.items():
        dataset = load_dataset(hf_path, split=split)
        if LIMIT:
            dataset = dataset.select(range(LIMIT))
        print(f"[{dataset_key}] {hf_path} -> {len(dataset)} problems")

        for template_name, prompt_template in PROMPT_TEMPLATES.items():
            run_name = f"{MODEL_KEY}/{dataset_key}/{template_name}"
            filename = os.path.join(RESULTS_DIR, f"results_{MODEL_KEY}_{dataset_key}_{template_name}{RESULT_SUFFIX}.json")
            if SKIP_IF_EXISTS and not LIMIT and os.path.exists(filename):
                prev = json.load(open(filename, encoding="utf-8"))
                # Chỉ skip khi file có đủ bài (tránh dùng nhầm file smoke test LIMIT=10 còn sót lại)
                if len(prev) == len(dataset):
                    summary[run_name] = sum(r["pass@k"] for r in prev) / len(prev)
                    print(f"Skip {run_name} (có sẵn {filename}) -> {summary[run_name]:.4f}")
                    continue
                print(f"Bỏ file cũ {filename} ({len(prev)}/{len(dataset)} bài) và chạy lại")
            summary[run_name] = evaluate_dataset(dataset, dataset_key, template_name, prompt_template)



########## hoangnh39/qwen2.5-coder-7b-opencode-hq10k-3ep@epoch-1 -> qwen_ft_opencode_hq10k_3ep_epoch1 (active adapter: epoch_1) ##########


README.md:   0%|          | 0.00/6.52k [00:00<?, ?B/s]

openai_humaneval/test-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 83.9kB            

openai_humaneval/test-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

...

[qwen_ft_opencode_hq10k_3ep_epoch2][humaneval_plus][few_shot] Avg pass@1: 0.7183 - Avg time: 34.39s/it
Results saved to /kaggle/working/data/results_qwen_ft_opencode_hq10k_3ep_epoch2_humaneval_plus_few_shot_n5.json

########## hoangnh39/qwen2.5-coder-7b-opencode-hq10k-3ep@epoch-3 -> qwen_ft_opencode_hq10k_3ep_epoch3 (active adapter: epoch_3) ##########
[humaneval] openai/openai_humaneval -> 164 problems

===== Model [qwen_ft_opencode_hq10k_3ep_epoch3] | Dataset [humaneval] | Template [few_shot] | Format [chatml] =====


 20%|█▉        | 32/164 [12:36<43:44, 19.88s/it]<string>:28: RuntimeWarning: The iteration is not making good progress, as measured by the 
 improvement from the last ten iterations.
100%|██████████| 164/164 [1:34:08<00:00, 34.44s/it]


[qwen_ft_opencode_hq10k_3ep_epoch3][humaneval][few_shot] Avg pass@1: 0.7598 - Avg time: 33.06s/it
Results saved to /kaggle/working/data/results_qwen_ft_opencode_hq10k_3ep_epoch3_humaneval_few_shot_n5.json
[humaneval_plus] evalplus/humanevalplus -> 164 problems

===== Model [qwen_ft_opencode_hq10k_3ep_epoch3] | Dataset [humaneval_plus] | Template [few_shot] | Format [chatml] =====


100%|██████████| 164/164 [1:36:48<00:00, 35.42s/it]

[qwen_ft_opencode_hq10k_3ep_epoch3][humaneval_plus][few_shot] Avg pass@1: 0.7061 - Avg time: 32.72s/it
Results saved to /kaggle/working/data/results_qwen_ft_opencode_hq10k_3ep_epoch3_humaneval_plus_few_shot_n5.json


## 7. Summary & So sánh với base / instruct

In [8]:
print(f"{'Run':60s} pass@{K_VALUE}")
print("-" * 72)
for run_name, score in summary.items():
    print(f"{run_name:60s} {score:.4f}")
save_results(summary, os.path.join(RESULTS_DIR, f"summary_qwen_ft_opencode_hq10k_3ep{RESULT_SUFFIX}.json"))

print(f"\n=== So sánh few-shot (fine-tuned: n={NUM_SAMPLES}; baseline: n=1) ===")
header = f"{'Benchmark':16s} {'base':>7s} {'instruct':>9s}" + "".join(f"{r:>10s}" for r in FT_REVISIONS) + "   Δ best vs base"
print(header)
for ds in DATASETS:
    base_s = BASELINES.get(f"qwen_base/{ds}/few_shot")
    inst_s = BASELINES.get(f"qwen_instruct/{ds}/few_shot")
    fts = [summary.get(f"{model_key(r)}/{ds}/few_shot") for r in FT_REVISIONS]
    done = [x for x in fts if x is not None]
    if not done:
        continue
    cells = "".join(f"{x:10.4f}" if x is not None else f"{'-':>10s}" for x in fts)
    print(f"{ds:16s} {base_s:7.4f} {inst_s:9.4f}{cells}   {max(done) - base_s:+.4f}")

Run                                                          pass@1
------------------------------------------------------------------------
qwen_ft_opencode_hq10k_3ep_epoch1/humaneval/few_shot         0.7866
qwen_ft_opencode_hq10k_3ep_epoch1/humaneval_plus/few_shot    0.7354
qwen_ft_opencode_hq10k_3ep_epoch2/humaneval/few_shot         0.7744
qwen_ft_opencode_hq10k_3ep_epoch2/humaneval_plus/few_shot    0.7183
qwen_ft_opencode_hq10k_3ep_epoch3/humaneval/few_shot         0.7598
qwen_ft_opencode_hq10k_3ep_epoch3/humaneval_plus/few_shot    0.7061

=== So sánh few-shot (fine-tuned: n=5; baseline: n=1) ===
Benchmark           base  instruct   epoch-1   epoch-2   epoch-3   Δ best vs base
humaneval         0.7683    0.8780    0.7866    0.7744    0.7598   +0.0183
humaneval_plus    0.6951    0.8232    0.7354    0.7183    0.7061   +0.0403


## 8. (Tuỳ chọn) Xem các case fail

```python
r = json.load(open(f"{RESULTS_DIR}/results_{MODEL_KEY}_humaneval_plus_few_shot.json"))
fails = [x for x in r if x["pass@k"] == 0]
print(len(fails), "fail")
print(fails[0]["task_id"]); print(fails[0]["samples"][0]["raw"])
```